# 03 — Does the relationship hold?

Testing the statement:

> **Companies that already had a Wikipedia article before going public are more
> underpriced.**

`underpricing = (first_close − offer_price) / offer_price × 100`

In [1]:
import json, math, collections
import numpy as np, pandas as pd
import pipeline as P

src = P.OUT / "panel_refined.json"
if not src.exists():
    src = P.OUT / "panel.json"
rows = json.loads(src.read_text())
df = pd.DataFrame(rows)
if "wiki_refined" not in df.columns:
    df["wiki_refined"] = df["wiki"].where(df.wiki_status == "accept")
print(f"raw panel: {len(df)}  (source: {src.name})")
df.head(3)[["ticker", "ipo_date", "offer_price", "close", "underpricing", "wiki", "wiki_status"]]

raw panel: 899  (source: panel_refined.json)


,ticker,ipo_date,offer_price,close,underpricing,wiki,wiki_status
0,TRVN,2014-01-31,7.0,6.50,-7.142857,0.0,accept
1,QURE,2014-02-05,17.0,14.61,-14.058826,0.0,accept
2,LADR,2014-02-06,17.0,16.99,-0.058825,0.0,ambiguous


## Cleaning

The raw panel has a standard deviation of **249 pp**, which is impossible for
first-day IPO returns (the real figure is ~40 pp). Two screens fix it, and the
second one is the important one.

1. **Offer price >= $5** — a conventional screen. Also removes the
   nano-caps where prospectus parsing is least reliable, and direct listings
   whose "reference price" is not an offer price at all.
2. **No post-IPO split** — Yahoo's `close` series is split-adjusted, and its
   split history for micro-caps is *incomplete*, so un-adjusting is impossible.
   Verified directly: E-Home Household reports seven reverse splits and still
   lands 10× too high; reAlpha reports one 1:25 split when it clearly had more.
   Split-adjusted rows carry sd = 421 pp against 133 pp for the rest.

Then winsorize at 1/99 so genuine large pops are retained without dominating.

**This second screen is a look-ahead restriction** — reverse splits happen to
firms that later collapse, so the retained sample skews toward survivors on top
of the survivorship already present in free price data. It is recorded as a
limitation rather than corrected, because correcting it needs CRSP.

In [2]:
n0 = len(df)
df = df[df.underpricing.notna()]

# Screen 1: offer price >= $5. A conventional screen, and it drops
# the nano-cap listings where prospectus parsing is least reliable.
df = df[(df.offer_price >= 5) & (df.offer_price <= 500)].copy()
print(f"offer price in [$5, $500]      : {len(df)}/{n0}")

# Screen 2: no post-IPO split. Yahoo's `close` is split-adjusted, and its
# split history for micro-caps is demonstrably INCOMPLETE, so the series
# cannot be reliably un-adjusted. E-Home Household (EJH) reports seven
# reverse splits and still lands 10x too high; reAlpha (AIRE) reports one
# 1:25 split when it plainly had more. Keeping these rows put the sample sd
# at 421 pp against 133 pp for unadjusted rows.
#
# This is a LOOK-AHEAD restriction: reverse splits happen to poor performers,
# so the retained sample skews toward firms that did not later collapse.
# Recorded as a limitation, not corrected.
pre = len(df)
df["split_factor"] = df.split_factor.fillna(1.0)
df = df[df.split_factor == 1.0].copy()
print(f"no post-IPO split adjustment   : {len(df)}/{pre}")

lo, hi = df.underpricing.quantile([0.01, 0.99])
df["up_w"] = df.underpricing.clip(lo, hi)
print(f"winsorized at [{lo:.1f}, {hi:.1f}] pp")
print(f"\nunderpricing: mean {df.up_w.mean():+.2f}  median {df.up_w.median():+.2f}  "
      f"sd {df.up_w.std():.2f}")
print("(US IPO benchmark: mean ~18%, median ~10%, sd ~40% -- this is in range)")

offer price in [$5, $500]      : 766/899
no post-IPO split adjustment   : 602/766
winsorized at [-35.0, 139.3] pp

underpricing: mean +18.71  median +10.08  sd 30.41
(US IPO benchmark: mean ~18%, median ~10%, sd ~40% -- this is in range)


## The main test — difference in means

This is the statement, tested directly. Welch's t-test (unequal variances),
restricted to rows the matcher accepted.

In [3]:
from scipy import stats

# Primary specification: matcher-accepted codings, plus rejects as genuine
# zeros (no plausible company article found). Ambiguous rows are held out and
# bounded in the robustness section below.
# Primary: matcher-accepted codings, rejects as genuine zeros, plus the
# human-adjudicated rows returned in review_verdicts.csv.
human_done = df.get("wiki_verdict_source", pd.Series(index=df.index, dtype=object)) == "human"
d = df[df.wiki_status.isin(["accept", "reject"]) | human_done].copy()
d["wiki"] = d.wiki_refined.where(human_done, d.wiki).fillna(0).astype(int)
print(f"primary sample: {len(d)}  (accept {(d.wiki_status=='accept').sum()}, "
      f"reject {(d.wiki_status=='reject').sum()}, human {int(human_done.sum())})")
a = d[d.wiki == 1].up_w
b = d[d.wiki == 0].up_w

t, p = stats.ttest_ind(a, b, equal_var=False)
se = math.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
diff = a.mean() - b.mean()

print(f"Wikipedia = 1 : n={len(a):4d}  mean underpricing {a.mean():+7.2f}%")
print(f"Wikipedia = 0 : n={len(b):4d}  mean underpricing {b.mean():+7.2f}%")
print(f"\ndifference     : {diff:+.2f} pp")
print(f"std error      : {se:.2f}")
print(f"t-statistic    : {t:+.2f}")
print(f"p-value        : {p:.4f}")
print(f"95% CI         : [{diff-1.96*se:+.2f}, {diff+1.96*se:+.2f}] pp")
print()

primary sample: 529  (accept 200, reject 313, human 16)
Wikipedia = 1 : n= 128  mean underpricing  +22.63%
Wikipedia = 0 : n= 401  mean underpricing  +17.08%

difference     : +5.55 pp
std error      : 3.26
t-statistic    : +1.70
p-value        : 0.0907
95% CI         : [-0.85, +11.94] pp



## With controls

We can afford `vc` (Ritter), `tech` (SIC 3570–3579 / 7370–7379 / 3661–3674,
from EDGAR), `log(proceeds)` (offer price × shares offered, from the
prospectus cover) and year fixed effects.

We cannot afford underwriter reputation, share overhang, or a pre-IPO news
count — all three normally matter for underpricing and plausibly correlate
with Wikipedia presence, so this estimate retains confounding a fuller
specification would remove.

In [4]:
import statsmodels.formula.api as smf

TECH_SIC = lambda s: bool(s) and (
    3570 <= int(s) <= 3579 or 7370 <= int(s) <= 7379 or 3661 <= int(s) <= 3674)

d = d.copy()
d["tech"] = d.sic.apply(lambda s: int(TECH_SIC(s)) if str(s).isdigit() else 0)
d["vc"] = d.vc.fillna(0).astype(int)
d["proceeds"] = d.offer_price * d.shares_offered
d["log_proceeds"] = np.log(d.proceeds.where(d.proceeds > 0))
d["year"] = d.year.astype(str)

est = d.dropna(subset=["log_proceeds"])
m = smf.ols("up_w ~ wiki + vc + tech + log_proceeds + C(year)", data=est).fit(cov_type="HC1")
print(m.summary().tables[1])
print(f"\nn = {int(m.nobs)}   adj R2 = {m.rsquared_adj:.3f}")
ci = m.conf_int().loc["wiki"]
print(f"\nWikipedia coefficient: {m.params['wiki']:+.2f} pp  "
      f"(SE {m.bse['wiki']:.2f}, p={m.pvalues['wiki']:.4f})")
print(f"95% CI: [{ci[0]:+.2f}, {ci[1]:+.2f}]")

                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept         -70.0597     17.670     -3.965      0.000    -104.693     -35.426
C(year)[T.2015]     6.6810      6.355      1.051      0.293      -5.774      19.136
C(year)[T.2016]     1.1496      6.879      0.167      0.867     -12.334      14.633
C(year)[T.2017]     1.1077      6.496      0.171      0.865     -11.625      13.840
C(year)[T.2018]     6.7175      5.581      1.204      0.229      -4.220      17.655
C(year)[T.2019]     6.2436      6.296      0.992      0.321      -6.096      18.583
C(year)[T.2020]    23.6511      6.377      3.709      0.000      11.152      36.151
C(year)[T.2021]     4.6763      5.151      0.908      0.364      -5.419      14.772
C(year)[T.2022]   -11.0907      5.332     -2.080      0.038     -21.541      -0.641
C(year)[T.2023]     2.8579      7.037      0.406      0.685     -10.934     

## Robustness to the ambiguous cases

The matcher escalated some firms rather than guessing. Bounding the result by
coding them all 1 and all 0 shows whether the conclusion depends on them.

In [5]:
def diff_means(frame):
    a, b = frame[frame.wiki == 1].up_w, frame[frame.wiki == 0].up_w
    if len(a) < 2 or len(b) < 2:
        return None
    se = math.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
    dd = a.mean() - b.mean()
    return dd, se, dd/se, len(a), len(b)

variants = {}
# Primary: matcher-accepted codings plus rejects as genuine zeros. Ambiguous
# rows are excluded here and bounded below.
hd = df.get("wiki_verdict_source", pd.Series(index=df.index, dtype=object)) == "human"
base = df[df.wiki_status.isin(["accept", "reject"]) | hd].copy()
base["wiki"] = base.wiki_refined.where(hd, base.wiki).fillna(0).astype(int)
variants["PRIMARY (accept+reject)"] = base
variants["+ auto-adjudicated"] = (
    lambda t: t.assign(wiki=t.wiki_refined.astype(int))
)(df[df.wiki_refined.notna()])
amb1 = df[df.wiki_status != "api_error"].copy()
amb1["wiki"] = amb1.wiki.fillna(0)
amb1.loc[amb1.wiki_status == "ambiguous", "wiki"] = 1
variants["+ ambiguous all -> 1"] = amb1
amb0 = df[df.wiki_status != "api_error"].copy()
amb0["wiki"] = amb0.wiki.fillna(0)
amb0.loc[amb0.wiki_status == "ambiguous", "wiki"] = 0
variants["+ ambiguous all -> 0"] = amb0

print(f"{'specification':26s} {'n1':>5s} {'n0':>5s} {'diff':>8s} {'SE':>7s} {'t':>7s}")
print("-" * 64)
for k, v in variants.items():
    r = diff_means(v)
    if r:
        dd, se, tt, n1, n0 = r
        print(f"{k:26s} {n1:5d} {n0:5d} {dd:+8.2f} {se:7.2f} {tt:+7.2f}")

specification                 n1    n0     diff      SE       t
----------------------------------------------------------------
PRIMARY (accept+reject)      128   401    +5.55    3.26   +1.70
+ auto-adjudicated           180   418    +5.37    2.77   +1.94
+ ambiguous all -> 1         214   388    +5.00    2.59   +1.93
+ ambiguous all -> 0         125   477    +4.57    3.25   +1.40


## Also: unwinsorized, and by era

In [6]:
raw = base
a, b = raw[raw.wiki == 1].underpricing, raw[raw.wiki == 0].underpricing
se = math.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))
print(f"unwinsorized: diff {a.mean()-b.mean():+.2f} pp, SE {se:.2f}, "
      f"t {(a.mean()-b.mean())/se:+.2f}")

print("\nby period:")
for label, lo_y, hi_y in [("2014-2018", 2014, 2018), ("2019-2024", 2019, 2024)]:
    sub = raw[(raw.year.astype(int) >= lo_y) & (raw.year.astype(int) <= hi_y)]
    r = diff_means(sub)
    if r:
        dd, se, tt, n1, n0 = r
        print(f"  {label}  n={n1+n0:4d}  diff {dd:+7.2f} pp  t {tt:+5.2f}")

unwinsorized: diff +10.05 pp, SE 6.84, t +1.47

by period:
  2014-2018  n= 162  diff   +5.02 pp  t +1.04
  2019-2024  n= 367  diff   +6.06 pp  t +1.44


## Verdict

See the printed output above. Interpretation guidance:

- The **sign** of the difference is the primary result. The statement predicts
  positive.
- With this sample size the test is underpowered against an effect of a few
  percentage points, so a confidence interval that includes zero is the
  expected outcome a substantial fraction of the time **even if the effect is
  real**.
- This is a **descriptive difference between two groups**, not a causal
  estimate. Nothing here identifies what Wikipedia does; the design has no
  instrument and no natural experiment.
- Known biases here: survivorship in free price data (delisted
  firms are absent), an incomplete control set (no `top_tier`, `overhang`,
  `log_news`), and rule-(1)-only matching, which misclassifies some Wikipedia
  firms as non-Wikipedia and **attenuates the estimate toward zero**.